# Machine Learning-Based Prediction of 30-Day Hospital Readmission in Diabetic Patients

## Feature Engineering

In [55]:
import pandas as pd
import numpy as np

from scipy.stats import chi2_contingency

In [56]:
df = pd.read_csv("../data/2. interim/cleaned.csv")

In [57]:
df.shape

(101766, 47)

In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 47 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   patient_nbr               101766 non-null  int64 
 1   race                      101766 non-null  object
 2   gender                    101766 non-null  object
 3   age                       101766 non-null  object
 4   admission_type            101766 non-null  object
 5   discharge_disposition     101766 non-null  object
 6   admission_source          101766 non-null  object
 7   time_in_hospital          101766 non-null  int64 
 8   payer_code                101766 non-null  object
 9   medical_specialty         101766 non-null  object
 10  num_lab_procedures        101766 non-null  int64 
 11  num_procedures            101766 non-null  int64 
 12  num_medications           101766 non-null  int64 
 13  number_outpatient         101766 non-null  int64 
 14  numb

### Convert Target Column

In [59]:
# Check unique values

print(df['readmitted'].value_counts())

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64


In [60]:
# Convert to binary
df['readmitted_binary'] = df['readmitted'].apply(
    lambda x: 1 if x == '<30' else 0
)

In [61]:
# Verify conversion
df['readmitted_binary'].value_counts()

readmitted_binary
0    90409
1    11357
Name: count, dtype: int64

In [62]:
df['readmitted_binary'].value_counts(normalize=True)

readmitted_binary
0    0.888401
1    0.111599
Name: proportion, dtype: float64

### Convert Drug Columns to Numeric

In [63]:
# No      → 0
# Steady  → 1
# Up      → 1
# Down    → 1

# Because the important question is was this drug given or not?

In [64]:
drug_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton',
    'insulin', 'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone',
    'metformin-pioglitazone'
]

missing_drug_cols = sorted(set(drug_cols) - set(df.columns))
if missing_drug_cols:
    raise ValueError(f'Missing expected medication columns: {missing_drug_cols}')

In [65]:
drug_mapping = {
    'No': 0,
    'Steady': 1,
    'Up': 1,
    'Down': 1
}

mapped_drugs = df[drug_cols].apply(
    lambda col: col.map(drug_mapping)
)

if mapped_drugs.isna().any().any():
    raise ValueError(
        "Unexpected or missing medication status found."
    )

df[drug_cols] = mapped_drugs.astype('int8')

### Creating Some Features
- `insulin_flag` is a binary version of insulin status.
- `oral_med_flag` indicates any non-insulin drug recorded.

#### Feature 1: Total Diabetes Drugs Given

In [66]:
df['total_diabetes_drugs'] = df[drug_cols].sum(axis=1)

# How many different diabetes drugs patient is taking.

#### Feature 2: Insulin Flag

In [67]:
df['insulin_flag'] = df['insulin']

# Insulin is very important indicator of severity.

#### Feature 3: Oral Medication Flag

In [68]:
oral_drugs = [col for col in drug_cols if col != 'insulin']
df['oral_med_flag'] = (df[oral_drugs].sum(axis=1) > 0).astype('int8')

### Medication Change Behavior

In [69]:
# Converting change column from categorical to numeric

df['change'].value_counts()

change
No    54755
Ch    47011
Name: count, dtype: int64

In [70]:
df['change'] = df['change'].map({'No':0,'Ch':1})

In [71]:
df['change'].value_counts()

change
0    54755
1    47011
Name: count, dtype: int64

In [72]:
# Converting diabetesMed from categorical to numeric column

df['diabetesMed'].value_counts()

diabetesMed
Yes    78363
No     23403
Name: count, dtype: int64

In [73]:
df['diabetesMed'] = df['diabetesMed'].map({"Yes":1 , "No":0})

In [74]:
df['diabetesMed'].value_counts()

diabetesMed
1    78363
0    23403
Name: count, dtype: int64

### Create new column as `high_medication_count_10plus`

In [75]:
# A modelling cut-off chosen for this project; not a universal clinical definition.

df['high_medication_count_10plus'] = (df['num_medications'] >= 10).astype('int8')

### Remove Original Medication Columns and Target

In [76]:
df.drop(columns=drug_cols, inplace=True)

In [77]:
df.drop(columns=['readmitted'], inplace=True)

In [78]:
df.shape

(101766, 28)

### Handling age column

In [79]:
df['age'].value_counts()

age
[70-80)     26068
[60-70)     22483
[50-60)     17256
[80-90)     17197
[40-50)      9685
[30-40)      3775
[90-100)     2793
[20-30)      1657
[10-20)       691
[0-10)        161
Name: count, dtype: int64

In [80]:
# Convert age bands to approximate midpoint values.
# These values represent age-band midpoints, not exact patient ages.

df['age'] = df['age'].str.extract(r'(\d+)').astype(int) + 5

### Checking payer_code column

In [81]:
# Applying 'chi2' test and finding 'Cramér's V' on payer_code to find Strength of the column with target column

In [82]:
df['payer_code'].nunique()

18

In [83]:
# Create a cross-tabulation table
contingency_table = pd.crosstab(df['payer_code'], df['readmitted_binary'])

# Run the test
chi2, p, dof, ex = chi2_contingency(contingency_table)
print(f"P-value: {p}")

P-value: 2.286749750616221e-13


In [84]:
# 'chi2' is the value you got from your test
n = df.shape[0]  # total observations
min_dim = min(contingency_table.shape) - 1
cramers_v = np.sqrt(chi2 / (n * min_dim))

print(f"Cramér's V (Strength): {cramers_v}")

Cramér's V (Strength): 0.030997806585539092


In [85]:
# Payer Code

# Retain payer_code as a categorical feature in the primary model.
# Missing values have already been encoded as "Unknown".

# Do not create payer_grouped or drop payer_code based on
# chi-square tests or Cramér's V calculated on the full dataset.

# The contribution of payer_code will be evaluated through model validation using training data, without using held-out test labels for feature selection.

### Cleaning A1Cresult

In [86]:
print(df['A1Cresult'].value_counts(dropna=False))

# Do not ordinal-map A1Cresult; it will be included in categorical_cols in notebook 05.

A1Cresult
NotTested    84748
>8            8216
Norm          4990
>7            3812
Name: count, dtype: int64


### Checking Categorical Columns (inspection only)

In [87]:
df['diag_1'].nunique()

12

In [88]:
df['diag_2'].nunique()

12

In [89]:
df['diag_3'].nunique()

12

In [90]:
df['admission_type'].nunique()

6

In [91]:
df['discharge_disposition'].nunique()

25

In [92]:
df['admission_source'].nunique()

15

In [93]:
df['medical_specialty'].nunique()

73

In [94]:
# None of tham need to be grouped , it can easly encoded using OneHotEncoding now

### Final-Data Checks and Saving Clean File

In [95]:
assert 'patient_nbr' in df.columns, 'patient_nbr is needed for group splitting.'
assert 'readmitted_binary' in df.columns
assert 'readmitted' not in df.columns
assert df['readmitted_binary'].isin([0, 1]).all()
assert df['patient_nbr'].notna().all()

print('Shape:', df.shape)
print('Target prevalence:', df['readmitted_binary'].mean())

df.to_csv('../data/3. processed/final_dataset.csv', index=False)

Shape: (101766, 28)
Target prevalence: 0.11159915885462728


In [96]:

# Create a separate eligible-discharge cohort for sensitivity analysis.
#
# Keep the full engineered dataset unchanged in df.
# Exclude encounters whose discharge disposition indicates death or hospice
# only from the separate eligible-discharge cohort.

terminal_or_hospice_dispositions = {
    'Expired',
    'Hospice / home',
    'Hospice / medical facility',
    'Expired at home. Medicaid only, hospice.',
    'Expired in a medical facility. Medicaid only, hospice.',
    'Expired, place unknown. Medicaid only, hospice.'
}

# Confirm which expected categories are present in the dataset.
observed_dispositions = set(
    df['discharge_disposition'].dropna().unique()
)

matched_dispositions = (
    terminal_or_hospice_dispositions & observed_dispositions
)

missing_expected_dispositions = (
    terminal_or_hospice_dispositions - observed_dispositions
)

print("Death/hospice categories found:")
print(sorted(matched_dispositions))

print("\nExpected categories not found:")
print(sorted(missing_expected_dispositions))

# Identify encounters to exclude from the sensitivity-analysis cohort.
exclude_mask = df['discharge_disposition'].isin(
    terminal_or_hospice_dispositions
)

print("\nCohort summary")
print("-" * 40)
print("Full dataset rows:", len(df))
print("Rows in death/hospice categories:", int(exclude_mask.sum()))
print("Eligible-discharge rows:", int((~exclude_mask).sum()))

# Create a separate copy; do not overwrite df.
df_eligible_discharge = df.loc[~exclude_mask].copy()

# Save the eligible-discharge cohort separately.
df_eligible_discharge.to_csv(
    '../data/3. processed/eligible_discharge_dataset.csv',
    index=False
)

print("\nFull dataset remains unchanged:", len(df))
print("Eligible-discharge dataset shape:", df_eligible_discharge.shape)

print(
    "Full dataset 30-day readmission prevalence:",
    round(df['readmitted_binary'].mean(), 4)
)

print(
    "Eligible-discharge 30-day readmission prevalence:",
    round(df_eligible_discharge['readmitted_binary'].mean(), 4)
)

# Basic checks for the new cohort.
assert len(df_eligible_discharge) <= len(df)
assert 'patient_nbr' in df_eligible_discharge.columns
assert 'readmitted_binary' in df_eligible_discharge.columns
assert df_eligible_discharge['readmitted_binary'].isin([0, 1]).all()

assert not df_eligible_discharge[
    'discharge_disposition'
].isin(terminal_or_hospice_dispositions).any()

print("\nEligible-discharge cohort saved successfully.")


Death/hospice categories found:
['Expired', 'Expired at home. Medicaid only, hospice.', 'Expired in a medical facility. Medicaid only, hospice.', 'Hospice / home', 'Hospice / medical facility']

Expected categories not found:
['Expired, place unknown. Medicaid only, hospice.']

Cohort summary
----------------------------------------
Full dataset rows: 101766
Rows in death/hospice categories: 2423
Eligible-discharge rows: 99343

Full dataset remains unchanged: 101766
Eligible-discharge dataset shape: (99343, 28)
Full dataset 30-day readmission prevalence: 0.1116
Eligible-discharge 30-day readmission prevalence: 0.1139

Eligible-discharge cohort saved successfully.
